# Week 45, Day 5: Confidence: knowing how likely you are to be right

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w045/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A calibrated confidence

Write conf(x, T) from the formula, summing over the six coherences with equal prior probability. Apply it to 60,000 simulated trials and compare accuracy with mean confidence in five confidence bins from 0.5 to 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2009)
cohs = np.array([0, 0.032, 0.064, 0.128, 0.256, 0.512]); k = 6.0; n = 60000
c = r.choice(cohs, n); s = np.where(r.random(n) < 0.5, 1, -1); T = r.choice([0.1, 0.2, 0.4, 0.8], n)
x = k * c * s * T + np.sqrt(T) * r.normal(size=n)
choice = np.where(x >= 0, 1, -1)
correct = np.where(c == 0, r.random(n) < 0.5, choice == s)

def conf(x, T):
    num, den = 0.0, 0.0
    for cc in cohs:
        mu = k * cc * T
        lp = np.exp(-(x - mu) ** 2 / (2 * T)); lm = np.exp(-(x + mu) ** 2 / (2 * T))
        ...
    return num / den

cf = conf(x, T)
edges = np.linspace(0.5, 1, 6); b = np.clip(np.digitize(cf, edges) - 1, 0, 4)
acc_b = [correct[b == i].mean() for i in range(5)]; conf_b = [cf[b == i].mean() for i in range(5)]
for i in range(5):
    print(f"confidence {edges[i]:.1f} to {edges[i + 1]:.1f}: mean confidence {conf_b[i]:.3f}, accuracy {acc_b[i]:.3f}")
plt.plot(conf_b, acc_b, "o-"); plt.plot([0.5, 1], [0.5, 1], "k:"); plt.xlabel("confidence"); plt.ylabel("accuracy"); plt.show()

In [ ]:
import numpy as np
def test_calibrated():
    "in every confidence bin, accuracy matches mean confidence within 0.02"
    assert all(abs(acc_b[i] - conf_b[i]) < 0.02 for i in range(5))
def test_time():
    "the same evidence gives lower confidence after a longer time"
    assert conf(np.array([1.0]), np.array([0.8]))[0] < conf(np.array([1.0]), np.array([0.2]))[0]

test_calibrated()
test_time()
print("All checks passed")

### Exercise: Opting out

Offer the sure target on a random half of trials; opt out when offered and confidence is below 0.7. Report the opt-out rate at each coherence, and at 6.4% coherence compare accuracy on offered-but-declined trials with accuracy on trials without the offer.

In [ ]:
offered = r.random(n) < 0.5
out = ...
opt = {cc: out[(c == cc) & offered].mean() for cc in cohs}
waived = offered & ~out
acc_waived = ...
acc_unoffered = ...
print({f"{100 * cc:.1f}%": round(v, 2) for cc, v in opt.items()}, "opt-out rate")
print(f"6.4% coherence: accuracy when the sure target was declined {acc_waived:.3f}, when not offered {acc_unoffered:.3f}")

In [ ]:
import numpy as np
def test_coherence():
    "the sure target is chosen far more often at 0% than at 51.2% coherence"
    assert opt[0.0] > opt[0.512] + 0.3
def test_waive():
    "at 6.4% coherence, accuracy when the sure target was offered and declined beats accuracy when it was not offered"
    assert acc_waived > acc_unoffered + 0.05

test_coherence()
test_waive()
print("All checks passed")

### Exercise: The folded X

For each non-zero coherence, average confidence separately over correct and error trials.

In [ ]:
nz = cohs[1:]
conf_cor = np.array([...])
conf_err = np.array([...])
for cc, a_, b_ in zip(nz, conf_cor, conf_err):
    print(f"{100 * cc:5.1f}%: confidence when correct {a_:.3f}, when wrong {b_:.3f}")
plt.semilogx(100 * nz, conf_cor, "o-", label="correct"); plt.semilogx(100 * nz, conf_err, "o-", label="error")
plt.xlabel("coherence (%)"); plt.ylabel("mean confidence"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_correct():
    "confidence on correct trials rises with coherence"
    assert all(np.diff(conf_cor) > 0)
def test_error():
    "confidence on error trials falls with coherence"
    assert all(np.diff(conf_err) < 0)

test_correct()
test_error()
print("All checks passed")